# 🚀 ByteMe Spring Boot - Load Testing on Kaggle

> **LƯU Ý TRƯỚC KHI CHẠY (BẮT BUỘC):**
> 1. **Notebook Settings** (cột bên phải):
>    - **Accelerator:** Chọn **None** (CPU để có 4 vCPUs và 30GB RAM).
>    - **Internet:** Bật **ON** (để tải package qua apt).
> 2. Nhấn nút **+ Add Input** ở góc trên bên phải -> Chọn Dataset bạn đã upload từ thư mục `kaggle/dataset` (chứa `application.jar`, `schema.sql`, `seed.sql`).

In [ ]:
# Cell 1: Tìm Dataset và Khởi tạo Môi trường (Java 21, MySQL 8, k6)
import os, glob, subprocess

matches = glob.glob('/kaggle/input/**/application.jar', recursive=True)
if not matches:
    raise FileNotFoundError('Không tìm thấy application.jar trong /kaggle/input! Hãy bấm "+ Add Input" và chọn Dataset của bạn.')

DATASET_DIR = os.path.dirname(matches[0])
print(f'[INFO] Đã tìm thấy dataset tại: {DATASET_DIR}')

# 1. Cài đặt các gói hệ thống
!apt-get update -qq
!apt-get install -y -qq openjdk-21-jdk mysql-server openssl curl gnupg

# 2. Cài đặt k6 nếu chưa có
!gpg -k 2>/dev/null || true
!gpg --no-default-keyring --keyring /usr/share/keyrings/k6-archive-keyring.gpg --keyserver hkp://keyserver.ubuntu.com:80 --recv-keys C5AD17C747E3415A3642D57D77C6C491D6AC1D69 2>/dev/null || true
!echo "deb [signed-by=/usr/share/keyrings/k6-archive-keyring.gpg] https://dl.k6.io/deb stable main" | tee /etc/apt/sources.list.d/k6.list
!apt-get update -qq && apt-get install -y -qq k6

# 3. Khởi động MySQL và nạp Database
!service mysql start
!mysql -u root -e "CREATE DATABASE IF NOT EXISTS starter CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci; CREATE USER IF NOT EXISTS 'starter'@'localhost' IDENTIFIED BY 'local-development-only'; GRANT ALL PRIVILEGES ON starter.* TO 'starter'@'localhost'; FLUSH PRIVILEGES;"
!mysql -u starter -plocal-development-only starter < "{DATASET_DIR}/schema.sql"
!mysql -u starter -plocal-development-only starter < "{DATASET_DIR}/seed.sql"
print('-> MySQL và dữ liệu mẫu đã sẵn sàng.')

# 4. Sinh RSA Keys cho JWT
!mkdir -p /kaggle/working/keys
!openssl genpkey -algorithm RSA -out /kaggle/working/keys/private.pem -pkeyopt rsa_keygen_bits:2048 2>/dev/null
!openssl rsa -pubout -in /kaggle/working/keys/private.pem -out /kaggle/working/keys/public.pem 2>/dev/null
!chmod 600 /kaggle/working/keys/private.pem
print('-> Khóa RSA đã tạo thành công.')

# 5. Khởi động Spring Boot ngầm với cấu hình tối ưu tải
env = os.environ.copy()
env.update({
    'DB_URL': 'jdbc:mysql://localhost:3306/starter?useSSL=false&allowPublicKeyRetrieval=true&serverTimezone=UTC',
    'DB_USERNAME': 'starter',
    'DB_PASSWORD': 'local-development-only',
    'SERVER_PORT': '8080',
    'AUTH_JWT_PRIVATE_KEY_FILE': '/kaggle/working/keys/private.pem',
    'AUTH_JWT_PUBLIC_KEY_FILE': '/kaggle/working/keys/public.pem',
    'AUTH_LOGIN_EMAIL_LIMIT': '1000000',
    'AUTH_LOGIN_IP_LIMIT': '1000000',
    'AUTH_LOGIN_WINDOW_SECONDS': '900',
    'SPRING_DATASOURCE_HIKARI_MAXIMUM_POOL_SIZE': '50',
    'SERVER_TOMCAT_THREADS_MAX': '200'
})

log_file = open('/kaggle/working/server.log', 'w')
server_proc = subprocess.Popen(
    ['java', '-Xms2g', '-Xmx6g', '-jar', os.path.join(DATASET_DIR, 'application.jar')],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=env
)
print(f'-> Server đang khởi động ngầm (PID: {server_proc.pid})...')

In [ ]:
# Cell 2: Kiểm tra Healthcheck cho đến khi Server sẵn sàng
import time, requests

ready = False
for i in range(35):
    try:
        r = requests.get('http://localhost:8080/actuator/health', timeout=2)
        if r.status_code == 200:
            ready = True
            print('>>> SERVER ĐÃ SẴN SÀNG (HEALTH = UP) <<<')
            break
    except:
        pass
    time.sleep(2)
    print(f'Đang chờ server ({i+1}/35)...')

if ready:
    foods = requests.get('http://localhost:8080/api/foods').json()
    print(f'-> Đã kết nối DB thành công. Thực đơn hiện có: {len(foods)} món ăn.')
else:
    print('!!! Server khởi động thất bại. Log 30 dòng cuối:')
    with open('/kaggle/working/server.log') as f:
        print(''.join(f.readlines()[-30:]))

In [ ]:
# Cell 3: Tạo kịch bản k6 và tiến hành kiểm thử tải (50 -> 200 -> 500 VUs)
k6_script_content = '''import http from "k6/http";
import { check, sleep, group } from "k6";

export const options = {
  stages: [
    { duration: "30s", target: 50 },
    { duration: "1m",  target: 200 },
    { duration: "1m",  target: 500 },
    { duration: "30s", target: 0 },
  ],
  thresholds: {
    http_req_duration: ["p(95)<500", "p(99)<1200"],
    http_req_failed: ["rate<0.01"],
  },
};

const BASE_URL = "http://localhost:8080";

export function setup() {
  const res = http.post(`${BASE_URL}/api/auth/token`, JSON.stringify({
    email: "an.customer@byteme.example",
    password: "ByteMeDemo!2026"
  }), { headers: { "Content-Type": "application/json" } });
  return { token: res.status === 200 ? JSON.parse(res.body).accessToken : null };
}

export default function (data) {
  group("Public Endpoints (Catalog & Search)", function () {
    const resFoods = http.get(`${BASE_URL}/api/foods`);
    check(resFoods, { "GET /api/foods status is 200": (r) => r.status === 200 });
    
    const resSearch = http.get(`${BASE_URL}/api/foods/search?q=ph%E1%BB%9F`);
    check(resSearch, { "GET /api/foods/search status is 200": (r) => r.status === 200 });
  });

  if (data.token) {
    group("Protected Endpoints (Cart)", function () {
      const resCart = http.get(`${BASE_URL}/api/cart`, {
        headers: { Authorization: `Bearer ${data.token}` }
      });
      check(resCart, { "GET /api/cart status is 200": (r) => r.status === 200 });
    });
  }

  group("Auth Token Generation", function () {
    const resToken = http.post(`${BASE_URL}/api/auth/token`, JSON.stringify({
      email: "binh.customer@byteme.example",
      password: "ByteMeDemo!2026"
    }), { headers: { "Content-Type": "application/json" } });
    check(resToken, { "POST /api/auth/token status is 200": (r) => r.status === 200 });
  });

  sleep(0.1);
}
'''

with open('/kaggle/working/load_test_k6.js', 'w', encoding='utf-8') as f:
    f.write(k6_script_content)

print('Bắt đầu chạy k6 Load Test...')
!k6 run --out csv=/kaggle/working/k6_results.csv /kaggle/working/load_test_k6.js

In [ ]:
# Cell 4: Vẽ biểu đồ Response Time & Throughput
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('/kaggle/working/k6_results.csv')

req_duration = df[df['metric_name'] == 'http_req_duration'].copy()
req_duration['time'] = pd.to_datetime(req_duration['timestamp'], unit='s')
req_duration = req_duration.sort_values('time')

reqs = df[df['metric_name'] == 'http_reqs'].copy()
reqs['time'] = pd.to_datetime(reqs['timestamp'], unit='s')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Biểu đồ 1: Latency
ax1.plot(req_duration['time'], req_duration['metric_value'], label='Latency (ms)', color='#1f77b4', alpha=0.5, linewidth=0.8)
ax1.axhline(y=500, color='orange', linestyle='--', label='SLA P95 (500ms)')
ax1.axhline(y=1000, color='red', linestyle='--', label='Alert (1000ms)')
ax1.set_ylabel('Response Time (ms)')
ax1.set_title('ByteMe - API Response Time Under Load', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# Biểu đồ 2: Throughput RPS
reqs_resampled = reqs.set_index('time').resample('1s')['metric_value'].sum()
ax2.plot(reqs_resampled.index, reqs_resampled.values, label='Requests / Second (RPS)', color='#2ca02c', linewidth=1.5)
ax2.set_xlabel('Timeline')
ax2.set_ylabel('RPS')
ax2.set_title('Throughput (Requests Per Second)', fontsize=14, fontweight='bold')
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/load_test_report.png', dpi=300)
plt.show()
print('[SUCCESS] Đã lưu biểu đồ tại /kaggle/working/load_test_report.png')

In [ ]:
# Cell 5: Tổng kết các chỉ số hiệu năng (P50, P90, P95, P99, Max)
p50 = req_duration['metric_value'].quantile(0.50)
p90 = req_duration['metric_value'].quantile(0.90)
p95 = req_duration['metric_value'].quantile(0.95)
p99 = req_duration['metric_value'].quantile(0.99)
total_requests = len(req_duration)

print('=' * 45)
print('          TỔNG KẾT CHỈ SỐ HIỆU NĂNG          ')
print('=' * 45)
print(f'Tổng số requests đã gửi: {total_requests:,}')
print(f'P50 (Median):             {p50:.2f} ms')
print(f'P90:                     {p90:.2f} ms')
print(f'P95:                     {p95:.2f} ms')
print(f'P99:                     {p99:.2f} ms')
print(f'Max Latency:             {req_duration["metric_value"].max():.2f} ms')
print('=' * 45)